# Phase 8B — Context Signal Audit — adapted to your Phase-8 export

This version is adapted to the uploaded raw Phase-8 CSV.

The export contains:
- `query_id`
- `mask`
- `quality`
- `token_f1`
- `exact_match`
- `raw_text`
- token / latency / cost fields

and uses **64 masks per query**, i.e.

\[
2^6=64,
\]

so this run has **6 candidate evidence items**, not 4.

For the main subset signal we use

\[
\boxed{F(S)=\texttt{quality}}
\]

which, in this export, matches the configured token-F1 metric and is continuous enough to be much more informative than `exact_match`.


In [1]:

import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 180)



## 1. Load the Phase-8 raw table

The notebook is intentionally schema-tolerant.  
It looks for common column names and reports what is available.

For best results, the raw table should contain one row per `(task_id, mask, repeat)` and, if possible:

- `provider`
- `task_id`
- `family`
- `mask`
- `prompt` or rendered input
- `evidence` / `context`
- `answer`
- `gold`
- `correct`
- optional `logprob_gold`, `margin`, `confidence`
- optional `repeat_id`


In [3]:
# Put the CSV in the same folder as this notebook, or change this path.
DATA_PATH = "phase8B_rag_raw_results.csv"

def load_table(path):
    p = Path(path)
    suf = p.suffix.lower()
    if suf == ".csv":
        return pd.read_csv(p)
    if suf in {".parquet", ".pq"}:
        return pd.read_parquet(p)
    if suf in {".jsonl", ".ndjson"}:
        return pd.read_json(p, lines=True)
    if suf == ".json":
        return pd.read_json(p)
    raise ValueError(f"Unsupported extension: {suf}")

raw = load_table(DATA_PATH)

print("rows:", len(raw))
print("columns:", list(raw.columns))
display(raw.head())

# Infer number of evidence bits from the observed mask range.
MAX_MASK = int(pd.to_numeric(raw["mask"], errors="raise").max())
N_EVIDENCE = max(1, int(math.ceil(math.log2(MAX_MASK + 1))))
N_STATES = 2 ** N_EVIDENCE

print("max mask:", MAX_MASK)
print("inferred N_EVIDENCE:", N_EVIDENCE)
print("expected complete-cube states:", N_STATES)


rows: 256
columns: ['run_id', 'manifest_sha256', 'provider', 'requested_model', 'returned_model', 'query_id', 'dataset_index', 'question', 'gold_answer', 'mask', 'order', 'repeat', 'effort', 'raw_text', 'exact_match', 'token_f1', 'quality', 'context_token_proxy', 'input_tokens', 'output_tokens', 'latency_sec', 'provider_cost_usd', 'error']


,run_id,manifest_sha256,provider,requested_model,returned_model,query_id,dataset_index,question,gold_answer,mask,order,repeat,effort,raw_text,exact_match,token_f1,quality,context_token_proxy,input_tokens,output_tokens,latency_sec,provider_cost_usd,error
0,d9b869f617fd515b,c13d90f172281ff4228ee4afa692b8e8aa6737a6906336193948c6d0f8a0be13,local,Qwen/Qwen3-0.6B,Qwen/Qwen3-0.6B,5a855b5c5542997b5ce3ffeb,681,"What is the main industry of this town with a population of around 2,600 as of 2010 that has Mountain Valley High School attendees as residents?",papermaking industry,0,0,0,low,UNKNOWN,0.0,0.00,0.00,0,102,2,75.856878,0.0,NaN
1,d9b869f617fd515b,c13d90f172281ff4228ee4afa692b8e8aa6737a6906336193948c6d0f8a0be13,local,Qwen/Qwen3-0.6B,Qwen/Qwen3-0.6B,5a855b5c5542997b5ce3ffeb,681,"What is the main industry of this town with a population of around 2,600 as of 2010 that has Mountain Valley High School attendees as residents?",papermaking industry,1,1,0,low,Mountain Valley High School,0.0,0.00,0.00,41,150,5,113.243740,0.0,NaN
2,d9b869f617fd515b,c13d90f172281ff4228ee4afa692b8e8aa6737a6906336193948c6d0f8a0be13,local,Qwen/Qwen3-0.6B,Qwen/Qwen3-0.6B,5a855b5c5542997b5ce3ffeb,681,"What is the main industry of this town with a population of around 2,600 as of 2010 that has Mountain Valley High School attendees as residents?",papermaking industry,2,1,0,low,Mountain Valley High School,0.0,0.00,0.00,80,182,5,134.191197,0.0,NaN
3,d9b869f617fd515b,c13d90f172281ff4228ee4afa692b8e8aa6737a6906336193948c6d0f8a0be13,local,Qwen/Qwen3-0.6B,Qwen/Qwen3-0.6B,5a855b5c5542997b5ce3ffeb,681,"What is the main industry of this town with a population of around 2,600 as of 2010 that has Mountain Valley High School attendees as residents?",papermaking industry,3,2,0,low,"The main industry of the town is a papermaking industry, and the school is located in Metea Valley.",0.0,0.25,0.25,121,234,23,187.184164,0.0,NaN
4,d9b869f617fd515b,c13d90f172281ff4228ee4afa692b8e8aa6737a6906336193948c6d0f8a0be13,local,Qwen/Qwen3-0.6B,Qwen/Qwen3-0.6B,5a855b5c5542997b5ce3ffeb,681,"What is the main industry of this town with a population of around 2,600 as of 2010 that has Mountain Valley High School attendees as residents?",papermaking industry,4,1,0,low,Mountain Valley High School,0.0,0.00,0.00,89,190,5,144.571601,0.0,NaN


max mask: 63
inferred N_EVIDENCE: 6
expected complete-cube states: 64


In [4]:
ALIASES = {
    "task": ["query_id", "task_id", "qid", "id"],
    "mask": ["mask", "subset_mask", "context_mask"],
    "family": ["family", "task_family", "kind"],
    "provider": ["provider", "model", "backend"],
    "prompt": ["prompt", "rendered_prompt", "input_text", "request"],
    "context": ["context", "evidence", "selected_evidence", "support_text"],
    "gold": ["gold_answer", "gold", "target", "reference_answer", "answer_gold"],
    "answer": ["raw_text", "answer", "prediction", "response", "model_answer"],
    "binary_correct": ["exact_match", "correct", "is_correct", "success"],
    "score": ["quality", "token_f1"],
    "repeat": ["repeat", "repeat_id", "sample_id", "trial"],
    "logprob": ["logprob_gold", "gold_logprob", "target_logprob"],
    "margin": ["margin", "gold_margin"],
    "confidence": ["confidence", "gold_confidence", "prob_gold"],
}

def resolve_col(df, logical_name):
    for c in ALIASES[logical_name]:
        if c in df.columns:
            return c
    return None

def resolved_schema(df):
    return {k: resolve_col(df, k) for k in ALIASES}

schema = resolved_schema(raw)
display(pd.DataFrame([schema]))

PRIMARY_SCORE = schema["score"]
BINARY_SCORE = schema["binary_correct"]

print("primary continuous score:", PRIMARY_SCORE)
print("binary score:", BINARY_SCORE)


,task,mask,family,provider,prompt,context,gold,answer,binary_correct,score,repeat,logprob,margin,confidence
0,query_id,mask,None,provider,None,None,gold_answer,raw_text,exact_match,quality,repeat,None,None,None


primary continuous score: quality
binary score: exact_match



## 2. Mask coverage and within-task variation

This is the first hard check.

For \(n=4\), every complete task should have 16 unique masks:

\[
0,\dots,15.
\]

We also compute whether `correct` is constant across all masks.


In [5]:
def mask_variation_report(df, n_evidence=N_EVIDENCE):
    schema = resolved_schema(df)
    task_col = schema["task"]
    mask_col = schema["mask"]
    score_col = schema["score"]
    binary_col = schema["binary_correct"]
    provider_col = schema["provider"]

    if task_col is None or mask_col is None:
        raise ValueError("Need query_id/task_id and mask columns.")

    group_cols = [c for c in [provider_col, task_col] if c is not None]

    rows = []
    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))

        masks = pd.to_numeric(g[mask_col], errors="coerce").dropna().astype(int)
        row["n_rows"] = len(g)
        row["n_masks"] = masks.nunique()
        row["complete_cube"] = masks.nunique() == 2**n_evidence
        row["mask_min"] = masks.min() if len(masks) else np.nan
        row["mask_max"] = masks.max() if len(masks) else np.nan

        if score_col is not None:
            x = pd.to_numeric(g[score_col], errors="coerce").dropna().to_numpy(float)
            row["score_mean"] = float(x.mean()) if len(x) else np.nan
            row["score_std"] = float(x.std(ddof=0)) if len(x) else np.nan
            row["score_range"] = float(np.ptp(x)) if len(x) else np.nan
            row["score_unique"] = int(len(np.unique(np.round(x, 12)))) if len(x) else 0
            row["score_flat"] = bool(len(x) and np.ptp(x) < 1e-12)

        if binary_col is not None:
            x = pd.to_numeric(g[binary_col], errors="coerce").dropna().to_numpy(float)
            row["exact_mean"] = float(x.mean()) if len(x) else np.nan
            row["exact_std"] = float(x.std(ddof=0)) if len(x) else np.nan
            row["exact_flat"] = bool(len(x) and np.ptp(x) < 1e-12)

        rows.append(row)

    return pd.DataFrame(rows)

mask_report = mask_variation_report(raw)
display(mask_report)


,provider,query_id,n_rows,n_masks,complete_cube,mask_min,mask_max,score_mean,score_std,score_range,score_unique,score_flat,exact_mean,exact_std,exact_flat
0,local,5a855b5c5542997b5ce3ffeb,64,64,True,0,63,0.14434,0.174320,0.444444,13,False,0.00000,0.000000,True
1,local,5a860aa6554299211dda2a45,64,64,True,0,63,0.57224,0.344430,1.000000,4,False,0.12500,0.330719,False
2,local,5a90b1a15542990a984936ab,64,64,True,0,63,0.03125,0.173993,1.000000,2,False,0.03125,0.173993,False
3,local,5abcd77755429965836004ce,64,64,True,0,63,0.00000,0.000000,0.000000,1,True,0.00000,0.000000,True


## 3. Leakage audit: limitation of this export

This CSV **does not contain the rendered prompt or the selected evidence text**.  
`raw_text` is the model output, not the complete input prompt.

Therefore we can test:
- mask coverage;
- score variation;
- output behavior at `mask=0` vs full mask;
- whether the gold answer appears in the **model output**;

but we **cannot prove that mask=0 was rendered without evidence** from this CSV alone.

For a true prompt-leak audit, also export either:
- the rendered prompt, or
- the selected evidence/context string for every mask.


In [6]:
def endpoint_output_audit(df, n_evidence=N_EVIDENCE):
    schema = resolved_schema(df)
    task_col = schema["task"]
    mask_col = schema["mask"]
    answer_col = schema["answer"]
    gold_col = schema["gold"]
    score_col = schema["score"]
    binary_col = schema["binary_correct"]
    provider_col = schema["provider"]

    full_mask = 2**n_evidence - 1
    group_cols = [c for c in [provider_col, task_col] if c is not None]
    rows = []

    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))

        g0 = g[pd.to_numeric(g[mask_col], errors="coerce") == 0]
        gf = g[pd.to_numeric(g[mask_col], errors="coerce") == full_mask]

        row["has_mask0"] = len(g0) > 0
        row["has_full"] = len(gf) > 0

        if len(g0) and len(gf):
            if score_col:
                s0 = float(g0.iloc[0][score_col])
                sf = float(gf.iloc[0][score_col])
                row["score_empty"] = s0
                row["score_full"] = sf
                row["score_full_minus_empty"] = sf - s0

            if binary_col:
                row["exact_empty"] = float(g0.iloc[0][binary_col])
                row["exact_full"] = float(gf.iloc[0][binary_col])

            if answer_col:
                a0 = str(g0.iloc[0][answer_col]).strip()
                af = str(gf.iloc[0][answer_col]).strip()
                row["output_empty_equals_full"] = (a0 == af)
                row["output_empty"] = a0
                row["output_full"] = af

            if gold_col and answer_col:
                gold = str(g0.iloc[0][gold_col]).strip().lower()
                a0 = str(g0.iloc[0][answer_col]).strip().lower()
                row["gold_literal_in_empty_output"] = bool(gold and gold in a0)

        rows.append(row)

    return pd.DataFrame(rows)

endpoint_audit = endpoint_output_audit(raw)
display(endpoint_audit)


,provider,query_id,has_mask0,has_full,score_empty,score_full,score_full_minus_empty,exact_empty,exact_full,output_empty_equals_full,output_empty,output_full,gold_literal_in_empty_output
0,local,5a855b5c5542997b5ce3ffeb,True,True,0.0,0.0,0.0,0.0,0.0,True,UNKNOWN,UNKNOWN,False
1,local,5a860aa6554299211dda2a45,True,True,0.0,0.0,0.0,0.0,0.0,True,UNKNOWN,UNKNOWN,False
2,local,5a90b1a15542990a984936ab,True,True,0.0,0.0,0.0,0.0,0.0,True,UNKNOWN,UNKNOWN,False
3,local,5abcd77755429965836004ce,True,True,0.0,0.0,0.0,0.0,0.0,True,UNKNOWN,UNKNOWN,False



## 4. Richer \(F(S)\): aggregate repeats and continuous scores

Binary correctness is too coarse if a model's confidence changes without changing the final answer.

For repeated sampling:

\[
\hat p_{\text{correct}}(S)
=
\frac1R\sum_{r=1}^R \mathbf 1[\hat y_r=y^\star].
\]

For continuous provider scores, prefer:
- gold log-probability;
- gold-vs-runner-up margin;
- calibrated confidence.

The function below builds a subset surface from whichever signals exist.


In [7]:
def build_subset_surfaces(df):
    schema = resolved_schema(df)
    task_col = schema["task"]
    mask_col = schema["mask"]
    provider_col = schema["provider"]

    group_cols = [c for c in [provider_col, task_col, mask_col] if c is not None]
    named_aggs = {}

    if schema["score"] is not None:
        named_aggs["F"] = (schema["score"], "mean")
        named_aggs["n_repeats"] = (schema["score"], "size")

    if schema["binary_correct"] is not None:
        named_aggs["p_exact"] = (schema["binary_correct"], "mean")

    if schema["logprob"] is not None:
        named_aggs["mean_logprob_gold"] = (schema["logprob"], "mean")

    if schema["margin"] is not None:
        named_aggs["mean_margin"] = (schema["margin"], "mean")

    if schema["confidence"] is not None:
        named_aggs["mean_confidence"] = (schema["confidence"], "mean")

    out = df.groupby(group_cols, dropna=False).agg(**named_aggs).reset_index()
    return out

surfaces = build_subset_surfaces(raw)
display(surfaces.head(20))

surfaces.to_csv("phase8b_subset_surface_quality.csv", index=False)
print("saved: phase8b_subset_surface_quality.csv")


,provider,query_id,mask,F,n_repeats,p_exact
0,local,5a855b5c5542997b5ce3ffeb,0,0.000000,1,0.0
1,local,5a855b5c5542997b5ce3ffeb,1,0.000000,1,0.0
2,local,5a855b5c5542997b5ce3ffeb,2,0.000000,1,0.0
3,local,5a855b5c5542997b5ce3ffeb,3,0.250000,1,0.0
4,local,5a855b5c5542997b5ce3ffeb,4,0.000000,1,0.0
5,local,5a855b5c5542997b5ce3ffeb,5,0.210526,1,0.0
6,local,5a855b5c5542997b5ce3ffeb,6,0.000000,1,0.0
7,local,5a855b5c5542997b5ce3ffeb,7,0.235294,1,0.0
8,local,5a855b5c5542997b5ce3ffeb,8,0.000000,1,0.0
9,local,5a855b5c5542997b5ce3ffeb,9,0.190476,1,0.0


saved: phase8b_subset_surface_quality.csv



## 5. Endpoint contrast: does evidence matter at all?

For every signal \(F\):

\[
\Delta_{\text{full-empty}}
=
F(\text{full})-F(\emptyset).
\]

If this is approximately zero for nearly every task, do not proceed to spectral interpretation yet.


In [8]:
def endpoint_contrast(surface_df, signal_col="F", n_evidence=N_EVIDENCE):
    schema = resolved_schema(surface_df)
    task_col = schema["task"] or "query_id"
    mask_col = schema["mask"] or "mask"
    provider_col = schema["provider"]

    full_mask = 2**n_evidence - 1
    group_cols = [c for c in [provider_col, task_col] if c is not None]

    rows = []
    for keys, g in surface_df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = dict(zip(group_cols, keys))

        g0 = g[pd.to_numeric(g[mask_col], errors="coerce") == 0]
        gf = g[pd.to_numeric(g[mask_col], errors="coerce") == full_mask]

        if len(g0) and len(gf):
            f0 = float(g0.iloc[0][signal_col])
            ff = float(gf.iloc[0][signal_col])
            row.update({
                "empty": f0,
                "full": ff,
                "full_minus_empty": ff-f0,
            })
        rows.append(row)

    return pd.DataFrame(rows)

contrasts = endpoint_contrast(surfaces, "F")
display(contrasts)


,provider,query_id,empty,full,full_minus_empty
0,local,5a855b5c5542997b5ce3ffeb,0.0,0.0,0.0
1,local,5a860aa6554299211dda2a45,0.0,0.0,0.0
2,local,5a90b1a15542990a984936ab,0.0,0.0,0.0
3,local,5abcd77755429965836004ce,0.0,0.0,0.0



## 6. Decision gate

Proceed to Phase 10 only if at least one of these is true for a meaningful fraction of tasks:

\[
\operatorname{std}_S F(S)>0
\]

or

\[
F(\text{full})-F(\emptyset)\neq0.
\]

Otherwise redesign the tasks or scoring before spectral analysis.


## Important limitation of this run

The uploaded export has:

\[
\texttt{repeat}=0
\]

only, so each `(query, mask)` currently has one stochastic realization.

The continuous `quality/token_f1` signal is still useful, but estimating a probability such as

\[
P(\text{correct}\mid S)
\]

would require multiple repeats per mask.

A follow-up run with, for example, \(R=8\) or \(R=16\) repeats per mask would let us separate subset effects from sampling noise.
